# EQORE public data retrieval

Run all cells to fetch and validate the six public sources plus Green Button XML from the supplied EQORE brief. Each check requires nonempty real data and the expected columns, displays a sample, and writes a Parquet snapshot with provenance. Failures are recorded so the remaining sources can still run. The final assertion fails if any source did not pass.

From the repo root, install with `uv pip install -r requirements.txt`, then launch with `uv run --env-file .env jupyter lab site_data_sources.ipynb`. OpenEI uses the public `DEMO_KEY` for small samples unless you set `OPENEI_API_KEY`.

The final sections build the Option 1 site-intake record and flag missing engineering inputs. See [the supplied-materials mapping](docs/eqore_site_intake.md) for the fields derived from the plan sets. The Green Button test file is labeled sample throughout.

In [1]:
import json
import logging
from pathlib import Path
from time import perf_counter

import polars as pl
from IPython.display import display

from API.site_sources import SiteDataClient, save_snapshot
from API.site_record import build_site_record

# Enable API logs to inspect endpoint URLs, cache hits, and pagination.
# logging.basicConfig(level=logging.INFO)
# logging.getLogger("API").setLevel(logging.DEBUG)

client = SiteDataClient(cache_dir="data/raw")
REFRESH = True  # False reuses complete cached downloads.
output_dir = Path("data/processed")
datasets = {}
results = []

def check_source(name, fetch, columns):
    started = perf_counter()
    try:
        frame = fetch()
        assert frame.height > 0, "The provider returned zero records."
        required = set(columns) | {"source_url", "retrieved_at_utc", "geography", "data_kind"}
        assert required <= set(frame.columns), f"Missing columns: {required - set(frame.columns)}"
        save_snapshot(frame, output_dir / f"{name}.parquet")
        datasets[name] = frame
        results.append({
            "source": name, "status": "PASS", "rows": frame.height,
            "seconds": round(perf_counter() - started, 2), "detail": "",
        })
        print(f"{name}: PASS — {frame.height:,} source rows")
        display(frame.select(columns).head(3))
    except Exception as error:
        # Continue independent source checks, then fail the notebook at the summary.
        message = str(error)
        auth_error = name == "openei_urdb" and any(text in message for text in ("API_KEY", "HTTP 401", "HTTP 403"))
        results.append({
            "source": name, "status": "AUTH REQUIRED" if auth_error else "FAIL", "rows": 0,
            "seconds": round(perf_counter() - started, 2),
            "detail": f"{type(error).__name__}: {message}",
        })
        print(f"{name}: {results[-1]['status']} — {results[-1]['detail']}")

## 1. California AB 802

Fetch the official 2025 energy workbook. Standard building columns are added and purchased-grid electricity is converted from kBtu to kWh. Raw disclosure fields remain available, including estimated-energy and incomplete-meter-data flags. These are reported annual values. A record may represent multiple buildings.

[CEC program and downloads](https://www.energy.ca.gov/programs-and-topics/programs/building-energy-benchmarking-program)

In [2]:
check_source(
    "ca_ab802",
    lambda: client.california_benchmarking(year=2025, refresh=REFRESH),
    ["building_id", "address", "city", "data_year", "latitude", "longitude", "annual_electricity_kwh"],
)

Could not determine dtype for column 18, falling back to string


ca_ab802: PASS — 27,822 source rows


building_id,address,city,data_year,latitude,longitude,annual_electricity_kwh
str,str,str,i32,f64,f64,f64
"""43177992""","""161 nortech parkway""","""san jose""",2025,37.424153,-121.957977,143853.231429
"""85978324""","""170 baytech drive""","""san jose""",2025,37.420262,-121.952022,1.5091e6
"""83061992""","""180 baytech drive""","""san jose""",2025,37.420986,-121.951917,26138.715679


## 2. Massachusetts LBER

Fetch the public layer linked by the state's results viewer. The `year` argument denotes the energy year, so 2025 uses the 2026 annual-report layer. This example deliberately uses small pages to exercise pagination. Use `limit=None` for the complete layer, or `where="City = 'Cambridge'"` to filter on the server.

[Official results map](https://www.mass.gov/info-details/large-building-energy-reporting-results)

In [3]:
check_source(
    "ma_lber",
    lambda: client.massachusetts_lber(year=2025, limit=100, page_size=40),
    ["building_id", "address", "city", "data_year", "latitude", "longitude", "annual_electricity_kwh"],
)

ma_lber: PASS — 100 source rows


building_id,address,city,data_year,latitude,longitude,annual_electricity_kwh
str,str,str,i32,f64,f64,f64
"""87HCQV6X+VX2-86-100-87-101""","""1180 Innovation Way""","""Fall River""",2025,41.762125,-71.100125,8.4184e6
"""87J94FR3+756-70-48-59-43""","""2145 Roosevelt Av""","""Springfield""",2025,42.1408,-72.547031,null
"""87J926G6+5P3-33-23-33-21""","""201 College Highway""","""Southwick""",2025,42.025388,-72.788172,215221.165762


## 3. ComStock

Fetch modeled building characteristics for Middlesex County, MA, then use the first model's `bldg_id` to retrieve its full 15-minute profile. The model identifier links these two ComStock files. It does not identify a real street address. The selected release uses 2018 weather.

Timestamps are retained in the provider's local simulation clock. Preserve this convention when comparing profiles or aggregating energy. A modeling release date differs from its weather year.

[ComStock release documentation](https://natlabrockies.github.io/ComStock.github.io/docs/data.html)

In [4]:
def fetch_comstock_sample():
    metadata = client.comstock_metadata(state="MA", county="G2500170", refresh=REFRESH)
    assert metadata.height > 0, "No modeled buildings found in the county."
    save_snapshot(metadata, output_dir / "comstock_metadata.parquet")
    display(metadata.select("bldg_id", "in.comstock_building_type").head(3))
    return client.comstock_profile(
        state="MA", building_id=metadata["bldg_id"][0], refresh=REFRESH
    )

check_source(
    "comstock",
    fetch_comstock_sample,
    ["building_id", "timestamp", "out.electricity.total.energy_consumption"],
)

bldg_id,in.comstock_building_type
i64,str
217,"""RetailStripmall"""
217,"""RetailStripmall"""
217,"""RetailStripmall"""


comstock: PASS — 35,040 source rows


building_id,timestamp,out.electricity.total.energy_consumption
str,datetime[ms],f32
"""217""",2018-01-01 00:15:00,0.942535
"""217""",2018-01-01 00:30:00,0.998708
"""217""",2018-01-01 00:45:00,1.00615


## 4. OpenEI utility tariffs

Request candidate commercial rate definitions for Cambridge, MA, and San Jose, CA. The complete nested definition is stored in `rate_json`. Use `json.loads(frame["rate_json"][0])` to access demand charges, time-of-use schedules, and energy tiers. Check effective dates and customer applicability before using a candidate in a bill calculation.

`OPENEI_API_KEY` is optional for these small tests because the public demo key works. It has a low shared rate limit. A personal key is useful for larger pulls. Use `offset` to retrieve additional pages.

[URDB API documentation](https://openei.org/services/doc/rest/util_rates/)

In [5]:
check_source(
    "openei_urdb",
    lambda: pl.concat([
        client.tariffs(42.36, -71.09, limit=2),
        client.tariffs(37.33, -121.89, limit=2),
    ], how="vertical_relaxed"),
    ["label", "utility", "name", "sector", "rate_json", "geography"],
)

openei_urdb: PASS — 4 source rows


label,utility,name,sector,rate_json,geography
str,str,str,str,str,str
"""539f6b90ec4f024411ec9e1f""","""NSTAR Electric Company""","""Cambridge Electric Light Compa…","""Commercial""","""{""label"": ""539f6b90ec4f024411e…","""42.36,-71.09"""
"""539f6becec4f024411eca345""","""NSTAR Electric Company""","""Cambridge Electric Light Compa…","""Commercial""","""{""label"": ""539f6becec4f024411e…","""42.36,-71.09"""
"""539f6b25ec4f024411ec98f5""","""Pacific Gas & Electric Co""","""A-1 TOU (Option B, Poly-Phase)""","""Commercial""","""{""label"": ""539f6b25ec4f024411e…","""37.33,-121.89"""


## 5. National Grid Massachusetts hosting capacity

Read feeder records from the public ArcGIS service, including available hosting capacity in MW and operating voltage in kV. The geometry is retained as WGS84 Esri JSON. These are distribution-generation hosting-capacity indicators. A utility assessment is needed to determine battery charging capacity and site-specific interconnection requirements.

[National Grid service metadata](https://systemdataportal.nationalgrid.com/arcgis/rest/services/MASDP/MASDP_HostingCapacity/MapServer)

In [6]:
check_source(
    "national_grid_ma",
    lambda: client.national_grid_capacity(limit=20, page_size=7),
    ["Master_CDF", "Substation", "Operating_Voltage_kV", "HC_Available_MW", "Updated_Date"],
)

national_grid_ma: PASS — 20 source rows


Master_CDF,Substation,Operating_Voltage_kV,HC_Available_MW,Updated_Date
str,str,f64,f64,i64
"""04_04_101L1""","""CANDLESTREET101""",13.2,1.0,1770354000000
"""04_04_101L2""","""CANDLESTREET101""",13.2,0.324721,1770354000000
"""04_04_101L3""","""CANDLESTREET101""",13.2,0.869582,1770354000000


## 6. Microsoft building footprints

Use the published index to locate the zoom-9 tiles containing the two example coordinates. Stream five real polygons from each tile. The sample is taken from the beginning of each tile, which can span many kilometers. Selecting an exact building requires a subsequent spatial match.

`feature_geojson` retains the complete polygon and its original properties. Geometry comes from machine learning and source imagery, so `data_kind` is `inferred`. Source imagery dates vary. Credit Microsoft and observe the dataset's CDLA Permissive 2.0 license when redistributing.

[Dataset, download index, and license](https://github.com/microsoft/GlobalMLBuildingFootprints)

In [7]:
check_source(
    "microsoft_footprints",
    lambda: pl.concat([
        client.building_footprints(42.36, -71.09, limit=5, refresh=REFRESH),
        client.building_footprints(37.33, -121.89, limit=5),
    ]),
    ["quadkey", "geometry_type", "feature_geojson", "geography"],
)

microsoft_footprints: PASS — 10 source rows


quadkey,geometry_type,feature_geojson,geography
str,str,str,str
"""030233212""","""Polygon""","""{""type"": ""Feature"", ""propertie…","""UnitedStates quadkey 030233212"""
"""030233212""","""Polygon""","""{""type"": ""Feature"", ""propertie…","""UnitedStates quadkey 030233212"""
"""030233212""","""Polygon""","""{""type"": ""Feature"", ""propertie…","""UnitedStates quadkey 030233212"""


## 7. Green Button customer interval XML

The supplied brief includes Green Button for customer-authorized utility exports. Fetch the public nine-day test file and parse its 216 hourly intervals. This is sample data, separate from all candidate buildings. `energy_kwh` applies each meter's own unit multiplier. `average_kw` divides interval energy by duration. Epoch timestamps stay in UTC; no fixed local offset is applied.

The reader supports incremental Wh energy and rejects other units or cumulative registers. Flow direction, phase, commodity, quality codes, and a file hash remain available for review. No utility login is attempted. Replace the sample with an authorized local XML export using `client.green_button("data/customer/usage.xml")`.

[Public samples](https://archive.greenbuttondata.org/samples/) · [Unit scaling](https://www.greenbuttonalliance.org/poweroftenmultiplier) · [Developer resources](https://www.greenbuttonalliance.org/developer-resources)

In [8]:
check_source(
    "green_button",
    lambda: client.green_button(refresh=REFRESH),
    ["meter_reading_uri", "interval_start_utc", "duration_seconds", "energy_kwh", "average_kw", "data_kind"],
)

green_button: PASS — 216 source rows


meter_reading_uri,interval_start_utc,duration_seconds,energy_kwh,average_kw,data_kind
str,"datetime[μs, UTC]",i64,f64,f64,str
"""https://services.greenbuttonda…",2014-01-01 05:00:00 UTC,3600,0.273,0.273,"""sample"""
"""https://services.greenbuttonda…",2014-01-01 06:00:00 UTC,3600,0.273,0.273,"""sample"""
"""https://services.greenbuttonda…",2014-01-01 07:00:00 UTC,3600,0.273,0.273,"""sample"""


## Retrieval summary

A PASS means the check fetched nonempty provider records, found the expected columns, and wrote a Parquet snapshot. A failure remains visible and makes this notebook fail during automated execution. To retry a particular source, rerun setup and the source cells, then the summary.

The raw workbook, metadata, profiles, and index downloads are cached in `data/raw`. Complete snapshots are written atomically in `data/processed`. Cached download timestamps retain the original retrieval time.

In [9]:
summary = pl.from_dicts(results)
save_snapshot(summary, output_dir / "source_checks.parquet")
display(summary)
assert summary.height == 7 and summary["status"].eq("PASS").all(), "One or more sources did not pass. See the summary above."


source,status,rows,seconds,detail
str,str,i64,f64,str
"""ca_ab802""","""PASS""",27822,7.53,""""""
"""ma_lber""","""PASS""",100,0.73,""""""
"""comstock""","""PASS""",35040,12.9,""""""
"""openei_urdb""","""PASS""",4,1.41,""""""
"""national_grid_ma""","""PASS""",20,0.38,""""""
"""microsoft_footprints""","""PASS""",10,8.74,""""""
"""green_button""","""PASS""",216,0.44,""""""


## Consume the normalized building data

Combine the two disclosure sources using the common columns. Keep provenance and data quality flags available in the full snapshots. This table is a starting point for gathering site evidence. A ComStock model can supply an illustrative load shape once a suitable archetype has been selected.

In [10]:
building_columns = [
    "building_id", "address", "city", "state", "latitude", "longitude", "building_type",
    "floor_area_sqft", "annual_electricity_kwh", "data_year",
    "source_url", "retrieved_at_utc", "geography", "data_kind",
]
buildings = pl.concat([
    datasets["ca_ab802"].select(building_columns),
    datasets["ma_lber"].select(building_columns),
], how="vertical_relaxed")
save_snapshot(buildings, output_dir / "buildings.parquet")
display(
    buildings.filter(pl.col("annual_electricity_kwh") > 0)
    .select("address", "state", "building_type", "annual_electricity_kwh", "data_kind")
    .head(5)
)


address,state,building_type,annual_electricity_kwh,data_kind
str,str,str,f64,str
"""161 nortech parkway""","""CA""","""Other - Education""",143853.231429,"""reported"""
"""170 baytech drive""","""CA""","""Office""",1.5091e6,"""reported"""
"""180 baytech drive""","""CA""","""Office""",26138.715679,"""reported"""
"""150 baytech drive""","""CA""","""Office""",3.3143e6,"""reported"""
"""4435 fortran drive""","""CA""","""Worship Facility""",142191.284004,"""reported"""


## Check interval coverage before consumption

Keep meters separate. Inspect duplicates, gaps, overlaps, durations, flow direction, and quality before modeling a bill. The public sample is hourly. It cannot reveal the site's 15-minute peak demand. Customer exports can have different durations and quality flags. Select the appropriate import stream and phase before aggregation.

In [11]:
intervals = datasets["green_button"].sort("meter_reading_uri", "interval_start_utc")
coverage = (
    intervals.with_columns(
        previous_end=pl.col("interval_end_utc").shift(1).over("meter_reading_uri")
    )
    .group_by("meter_reading_uri", "flow_direction", "phase")
    .agg(
        intervals=pl.len(),
        first_utc=pl.col("interval_start_utc").min(),
        last_end_utc=pl.col("interval_end_utc").max(),
        min_duration_seconds=pl.col("duration_seconds").min(),
        max_duration_seconds=pl.col("duration_seconds").max(),
        duplicate_starts=pl.len() - pl.col("interval_start_utc").n_unique(),
        gaps=(pl.col("interval_start_utc") > pl.col("previous_end")).sum(),
        overlaps=(pl.col("interval_start_utc") < pl.col("previous_end")).sum(),
        readings_with_quality_codes=pl.col("reading_quality").ne("").sum(),
        total_energy_kwh=pl.col("energy_kwh").sum(),
    )
)
display(coverage)
save_snapshot(coverage, output_dir / "interval_coverage.parquet")
assert intervals.height == 216 and intervals["data_kind"].eq("sample").all()
assert coverage.select("duplicate_starts", "gaps", "overlaps").sum_horizontal().sum() == 0
assert abs(intervals["energy_kwh"][0] - 0.273) < 1e-9

meter_reading_uri,flow_direction,phase,intervals,first_utc,last_end_utc,min_duration_seconds,max_duration_seconds,duplicate_starts,gaps,overlaps,readings_with_quality_codes,total_energy_kwh
str,str,str,u32,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,u32,u32,u32,u32,f64
"""https://services.greenbuttonda…","""1""","""769""",216,2014-01-01 05:00:00 UTC,2014-01-10 05:00:00 UTC,3600,3600,0,0,0,0,199.563


## Create the Option 1 site-intake record

Explicitly select one reported MA building as a candidate. This is a public disclosure record, with an unverified customer/site match. The 25-field checklist comes from the supplied brief, installation photos, and electrical/site plans. Every available value carries provenance and `confidence=unreviewed`; missing values carry a request for the evidence needed.

The source locator identifies the disclosure ID, energy year, and normalized field. Original disclosure flags remain in `ma_lber.parquet`. Review them before interpreting energy values. Nearby tariffs, modeled loads, tile polygons, and feeders stay in separate source tables until an actual site match is confirmed. Sample-plan electrical ratings belong to those sample projects.

The CSV is editable and the Parquet preserves types for later analysis. The missing-input list gives the customer or engineer a concrete follow-up list. This is an intake checklist, not a permit approval or compliance score.

In [12]:
candidate = (
    buildings.filter(
        (pl.col("state") == "MA")
        & pl.col("building_id").is_not_null()
        & pl.col("address").is_not_null()
    )
    .head(1)
    .row(0, named=True)
)
site_record = build_site_record(candidate)
missing_inputs = site_record.filter("is_missing").select("site_id", "field", "unit", "input_to_request")
save_snapshot(site_record, output_dir / "site_record.parquet")
site_record.write_csv(output_dir / "site_record.csv")
missing_inputs.write_csv(output_dir / "site_missing_inputs.csv")
print(f"Candidate {site_record['site_id'][0]}: {candidate['address']}, {candidate['city']}")
print(f"{site_record.height - missing_inputs.height} fields need review; {missing_inputs.height} inputs are missing.")
display(site_record.select("field", "value", "unit", "status", "confidence"))
display(missing_inputs)

Candidate MA:87HCQV6X+VX2-86-100-87-101: 1180 Innovation Way, Fall River
9 fields need review; 16 inputs are missing.


field,value,unit,status,confidence
str,str,str,str,str
"""address""","""1180 Innovation Way""",null,"""needs_review""","""unreviewed"""
"""city""","""Fall River""",null,"""needs_review""","""unreviewed"""
"""state""","""MA""",null,"""needs_review""","""unreviewed"""
"""latitude""","""41.762125""","""degrees""","""needs_review""","""unreviewed"""
"""longitude""","""-71.100125""","""degrees""","""needs_review""","""unreviewed"""
…,…,…,…,…
"""property_boundaries""",null,null,"""missing""","""unknown"""
"""proposed_bess_location""",null,null,"""missing""","""unknown"""
"""measured_clearances_ft""",null,"""ft""","""missing""","""unknown"""


site_id,field,unit,input_to_request
str,str,str,str
"""MA:87HCQV6X+VX2-86-100-87-101""","""utility""",null,"""Request a recent utility bill"""
"""MA:87HCQV6X+VX2-86-100-87-101""","""tariff_name""",null,"""Confirm rate code and effectiv…"
"""MA:87HCQV6X+VX2-86-100-87-101""","""interval_data""",null,"""Request customer-authorized in…"
"""MA:87HCQV6X+VX2-86-100-87-101""","""service_voltage_v""","""V""","""Request service nameplate phot…"
"""MA:87HCQV6X+VX2-86-100-87-101""","""service_amperage_a""","""A""","""Request main service rating fr…"
…,…,…,…
"""MA:87HCQV6X+VX2-86-100-87-101""","""property_boundaries""",null,"""Request parcel record or surve…"
"""MA:87HCQV6X+VX2-86-100-87-101""","""proposed_bess_location""",null,"""Request marked site plan with …"
"""MA:87HCQV6X+VX2-86-100-87-101""","""measured_clearances_ft""","""ft""","""Request measured offsets to bo…"
